In [1]:
# Setup

import os
import json
import random
import subprocess

# Set seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Configuration
PROJECT_ID = "msda-llm-finetuning"
PROCESSED_BUCKET = "msda-llm-processed-data"
MODELS_BUCKET = "msda-llm-models"
WANDB_PROJECT = "msda-llm-finetuning"

# Output directory
OUTPUT_DIR = "/content/vast_training"
os.makedirs(f"{OUTPUT_DIR}/configs", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/data", exist_ok=True)

print(f"   Output: {OUTPUT_DIR}")

   Output: /content/vast_training


In [2]:
# Authenticate GCP and Download Data

from google.colab import auth
auth.authenticate_user()

!gcloud config set project {PROJECT_ID} --quiet

print("Downloading training data from GCS...\n")

# Download the split training files
!gsutil cp gs://{PROCESSED_BUCKET}/theory/theory_train_split.jsonl {OUTPUT_DIR}/data/
!gsutil cp gs://{PROCESSED_BUCKET}/code/code_train_split.jsonl {OUTPUT_DIR}/data/
!gsutil cp gs://{PROCESSED_BUCKET}/math/math_train_split.jsonl {OUTPUT_DIR}/data/

print("\nDownloaded:")
!ls -lh {OUTPUT_DIR}/data/

Updated property [core/project].

Copying gs://msda-llm-processed-data/theory/theory_train_split.jsonl...
/ [1 files][966.9 KiB/966.9 KiB]                                                
Operation completed over 1 objects/966.9 KiB.                                    
Copying gs://msda-llm-processed-data/code/code_train_split.jsonl...
- [1 files][ 35.9 MiB/ 35.9 MiB]                                                
Operation completed over 1 objects/35.9 MiB.                                     
Copying gs://msda-llm-processed-data/math/math_train_split.jsonl...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

- [1 files][237.0 MiB/237.0 MiB]                                                
Operation completed over 1 objects/237.0 MiB.                                    

Downloaded:
total 274M


In [3]:
# Sample and Split Data

def load_jsonl(filepath):
    """Load JSONL file."""
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def save_jsonl(data, filepath):
    """Save to JSONL file."""
    with open(filepath, 'w') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"   Saved: {filepath} ({len(data):,} examples)")

def sample_and_split(data, sample_size=None, val_ratio=0.1, seed=42):
    """Sample data (if needed) and split into train/val."""
    random.seed(seed)

    # Sample if needed
    if sample_size and len(data) > sample_size:
        data = random.sample(data, sample_size)
        print(f"   Sampled {sample_size:,} from {len(data):,}")

    # Shuffle
    random.shuffle(data)

    # Split
    val_size = int(len(data) * val_ratio)
    val_data = data[:val_size]
    train_data = data[val_size:]

    return train_data, val_data

print("=" * 60)
print("PROCESSING DATASETS")
print("=" * 60)

# Theory: Use ALL data, 10% validation
print("\nTHEORY:")
theory_data = load_jsonl(f"{OUTPUT_DIR}/data/theory_train_split.jsonl")
print(f"   Loaded: {len(theory_data):,} examples")
theory_train, theory_val = sample_and_split(theory_data, sample_size=None, val_ratio=0.1)
print(f"   Train: {len(theory_train):,} | Val: {len(theory_val):,}")

# Code: Sample 10K, 10% validation
print("\nCODE:")
code_data = load_jsonl(f"{OUTPUT_DIR}/data/code_train_split.jsonl")
print(f"   Loaded: {len(code_data):,} examples")
code_train, code_val = sample_and_split(code_data, sample_size=10000, val_ratio=0.1)
print(f"   Train: {len(code_train):,} | Val: {len(code_val):,}")

# Math: Sample 10K, 10% validation
print("\nMATH:")
math_data = load_jsonl(f"{OUTPUT_DIR}/data/math_train_split.jsonl")
print(f"   Loaded: {len(math_data):,} examples")
math_train, math_val = sample_and_split(math_data, sample_size=10000, val_ratio=0.1)
print(f"   Train: {len(math_train):,} | Val: {len(math_val):,}")

# Save processed files
print("\n" + "=" * 60)
print("SAVING PROCESSED FILES")
print("=" * 60)

print("\nTheory:")
save_jsonl(theory_train, f"{OUTPUT_DIR}/data/theory_train.jsonl")
save_jsonl(theory_val, f"{OUTPUT_DIR}/data/theory_val.jsonl")

print("\nCode:")
save_jsonl(code_train, f"{OUTPUT_DIR}/data/code_train.jsonl")
save_jsonl(code_val, f"{OUTPUT_DIR}/data/code_val.jsonl")

print("\nMath:")
save_jsonl(math_train, f"{OUTPUT_DIR}/data/math_train.jsonl")
save_jsonl(math_val, f"{OUTPUT_DIR}/data/math_val.jsonl")

# Clean up original files to save space
!rm {OUTPUT_DIR}/data/*_split.jsonl

print("\nFinal processed files:")
!ls -lh {OUTPUT_DIR}/data/

PROCESSING DATASETS

THEORY:
   Loaded: 1,492 examples
   Train: 1,343 | Val: 149

CODE:
   Loaded: 17,499 examples
   Sampled 10,000 from 10,000
   Train: 9,000 | Val: 1,000

MATH:
   Loaded: 124,304 examples
   Sampled 10,000 from 10,000
   Train: 9,000 | Val: 1,000

SAVING PROCESSED FILES

Theory:
   Saved: /content/vast_training/data/theory_train.jsonl (1,343 examples)
   Saved: /content/vast_training/data/theory_val.jsonl (149 examples)

Code:
   Saved: /content/vast_training/data/code_train.jsonl (9,000 examples)
   Saved: /content/vast_training/data/code_val.jsonl (1,000 examples)

Math:
   Saved: /content/vast_training/data/math_train.jsonl (9,000 examples)
   Saved: /content/vast_training/data/math_val.jsonl (1,000 examples)

Final processed files:
total 41M
-rw-r--r-- 1 root root  19M Dec  3 16:15 code_train.jsonl
-rw-r--r-- 1 root root 2.1M Dec  3 16:15 code_val.jsonl
-rw-r--r-- 1 root root  18M Dec  3 16:15 math_train.jsonl
-rw-r--r-- 1 root root 2.0M Dec  3 16:15 math_val.

In [4]:
# Create configs/models.json

models_config = {
    "falcon_h1_7b": {
        "hf_model_name": "unsloth/Falcon-H1-7B-Instruct",
        "model_type": "unsloth",
        "load_in_4bit": False,
        "dtype": "bfloat16",
        "max_seq_length": 2048,
        "category": "theory",
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05
    },
    "qwen2_5_7b_4bit": {
        "hf_model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        "model_type": "unsloth_4bit",
        "load_in_4bit": True,
        "dtype": "float16",
        "max_seq_length": 2048,
        "category": "theory",
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05
    },
    "seed_coder_8b_4bit": {
        "hf_model_name": "unsloth/Seed-Coder-8B-Instruct-bnb-4bit",
        "model_type": "unsloth_4bit",
        "load_in_4bit": True,
        "dtype": "float16",
        "max_seq_length": 2048,
        "category": "code",
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05
    },
    "acemath_7b": {
        "hf_model_name": "nvidia/AceMath-7B-Instruct",
        "model_type": "standard",
        "load_in_4bit": True,
        "dtype": "bfloat16",
        "max_seq_length": 2048,
        "category": "math",
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05
    },
    "qwen2_5_math_7b_4bit": {
        "hf_model_name": "unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit",
        "model_type": "unsloth_4bit",
        "load_in_4bit": True,
        "dtype": "float16",
        "max_seq_length": 2048,
        "category": "math",
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05
    }
}

with open(f"{OUTPUT_DIR}/configs/models.json", "w") as f:
    json.dump(models_config, f, indent=2)

print("Created configs/models.json")
print(json.dumps(models_config, indent=2))

Created configs/models.json
{
  "falcon_h1_7b": {
    "hf_model_name": "unsloth/Falcon-H1-7B-Instruct",
    "model_type": "unsloth",
    "load_in_4bit": false,
    "dtype": "bfloat16",
    "max_seq_length": 2048,
    "category": "theory",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05
  },
  "qwen2_5_7b_4bit": {
    "hf_model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "model_type": "unsloth_4bit",
    "load_in_4bit": true,
    "dtype": "float16",
    "max_seq_length": 2048,
    "category": "theory",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05
  },
  "seed_coder_8b_4bit": {
    "hf_model_name": "unsloth/Seed-Coder-8B-Instruct-bnb-4bit",
    "model_type": "unsloth_4bit",
    "load_in_4bit": true,
    "dtype": "float16",
    "max_seq_length": 2048,
    "category": "code",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05
  },
  "acemath_7b": {
    "hf_model_name": "nvidia/AceMath-7B-Instruct",
    "model_type": "standard",
    

In [5]:
# Create configs/training.json

training_config = {
    "wandb_project": "msda-llm-finetuning",
    "gcs_output_bucket": "gs://msda-llm-models/adapters",

    "training_args": {
        "falcon_h1_7b": {
            "num_train_epochs": 3,
            "per_device_train_batch_size": 4,
            "per_device_eval_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "learning_rate": 2e-4,
            "warmup_ratio": 0.1,
            "lr_scheduler_type": "cosine",
            "logging_steps": 5,
            "eval_steps": 25,
            "save_strategy": "epoch",
            "evaluation_strategy": "steps",
            "bf16": True
        },
        "qwen2_5_7b_4bit": {
            "num_train_epochs": 3,
            "per_device_train_batch_size": 4,
            "per_device_eval_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "learning_rate": 2e-4,
            "warmup_ratio": 0.1,
            "lr_scheduler_type": "cosine",
            "logging_steps": 5,
            "eval_steps": 25,
            "save_strategy": "epoch",
            "evaluation_strategy": "steps",
            "fp16": True
        },
        "seed_coder_8b_4bit": {
            "num_train_epochs": 2,
            "per_device_train_batch_size": 4,
            "per_device_eval_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "learning_rate": 2e-4,
            "warmup_ratio": 0.1,
            "lr_scheduler_type": "cosine",
            "logging_steps": 10,
            "eval_steps": 50,
            "save_strategy": "epoch",
            "evaluation_strategy": "steps",
            "fp16": True
        },
        "acemath_7b": {
            "num_train_epochs": 2,
            "per_device_train_batch_size": 4,
            "per_device_eval_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "learning_rate": 2e-4,
            "warmup_ratio": 0.1,
            "lr_scheduler_type": "cosine",
            "logging_steps": 10,
            "eval_steps": 50,
            "save_strategy": "epoch",
            "evaluation_strategy": "steps",
            "bf16": True
        },
        "qwen2_5_math_7b_4bit": {
            "num_train_epochs": 2,
            "per_device_train_batch_size": 4,
            "per_device_eval_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "learning_rate": 2e-4,
            "warmup_ratio": 0.1,
            "lr_scheduler_type": "cosine",
            "logging_steps": 10,
            "eval_steps": 50,
            "save_strategy": "epoch",
            "evaluation_strategy": "steps",
            "fp16": True
        }
    }
}

with open(f"{OUTPUT_DIR}/configs/training.json", "w") as f:
    json.dump(training_config, f, indent=2)

print("Created configs/training.json")

Created configs/training.json


In [6]:
# Create train.py (Main Training Script)

train_py = '''#!/usr/bin/env python3
"""
Phase 3: LoRA Fine-Tuning Script
Usage: python train.py --model_key falcon_h1_7b
"""

import os
import sys
import json
import argparse
from datetime import datetime

import torch
import wandb
from datasets import load_dataset, Dataset
from transformers import TrainingArguments
from trl import SFTTrainer


def load_configs(model_key):
    """Load configurations."""
    with open("configs/models.json") as f:
        models = json.load(f)
    with open("configs/training.json") as f:
        training = json.load(f)

    if model_key not in models:
        print(f"Unknown model_key: {model_key}")
        print(f"   Available: {list(models.keys())}")
        sys.exit(1)

    return models[model_key], training["training_args"][model_key], training


def load_model_and_tokenizer(model_cfg):
    """Load model and tokenizer based on type."""
    model_name = model_cfg["hf_model_name"]
    model_type = model_cfg["model_type"]
    max_seq_length = model_cfg["max_seq_length"]

    print(f"Loading: {model_name}")
    print(f"   Type: {model_type}")

    if model_type in ["unsloth", "unsloth_4bit"]:
        from unsloth import FastLanguageModel

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=model_cfg["load_in_4bit"],
            dtype=torch.bfloat16 if model_cfg["dtype"] == "bfloat16" else torch.float16,
        )

        model = FastLanguageModel.get_peft_model(
            model,
            r=model_cfg["lora_r"],
            lora_alpha=model_cfg["lora_alpha"],
            lora_dropout=model_cfg["lora_dropout"],
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=42,
        )
    else:
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

        if model_cfg["load_in_4bit"]:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
            )
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )
            model = prepare_model_for_kbit_training(model)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True,
            )

        lora_config = LoraConfig(
            r=model_cfg["lora_r"],
            lora_alpha=model_cfg["lora_alpha"],
            lora_dropout=model_cfg["lora_dropout"],
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.print_trainable_parameters()
    return model, tokenizer


def format_example(example):
    """Format conversation to text."""
    messages = example.get("messages", [])

    text_parts = []
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        if role == "system":
            text_parts.append(f"### System:\\n{content}")
        elif role == "user":
            text_parts.append(f"### User:\\n{content}")
        elif role == "assistant":
            text_parts.append(f"### Assistant:\\n{content}")

    return {"text": "\\n\\n".join(text_parts)}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_key", type=str, required=True)
    parser.add_argument("--output_dir", type=str, default="./outputs")
    args = parser.parse_args()

    model_key = args.model_key
    output_dir = os.path.join(args.output_dir, model_key)
    os.makedirs(output_dir, exist_ok=True)

    # Load configs
    model_cfg, train_cfg, global_cfg = load_configs(model_key)
    category = model_cfg["category"]

    # Initialize W&B
    run_name = f"{model_key}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    wandb.init(
        project=global_cfg["wandb_project"],
        name=run_name,
        config={
            "model_key": model_key,
            "model_name": model_cfg["hf_model_name"],
            "category": category,
            **train_cfg,
        },
        tags=[category, "lora", model_key],
    )

    print("=" * 60)
    print(f"Training: {model_key}")
    print(f"   Model: {model_cfg['hf_model_name']}")
    print(f"   Category: {category}")
    print(f"   Output: {output_dir}")
    print("=" * 60)

    # Load datasets
    print(f"\\nLoading {category} dataset...")
    train_data = load_dataset("json", data_files=f"data/{category}_train.jsonl", split="train")
    val_data = load_dataset("json", data_files=f"data/{category}_val.jsonl", split="train")

    print(f"   Train: {len(train_data):,} examples")
    print(f"   Val: {len(val_data):,} examples")

    # Format datasets
    train_data = train_data.map(format_example, remove_columns=train_data.column_names)
    val_data = val_data.map(format_example, remove_columns=val_data.column_names)

    # Load model
    model, tokenizer = load_model_and_tokenizer(model_cfg)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=train_cfg["num_train_epochs"],
        per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
        per_device_eval_batch_size=train_cfg["per_device_eval_batch_size"],
        gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
        learning_rate=train_cfg["learning_rate"],
        warmup_ratio=train_cfg["warmup_ratio"],
        lr_scheduler_type=train_cfg["lr_scheduler_type"],
        logging_steps=train_cfg["logging_steps"],
        eval_steps=train_cfg["eval_steps"],
        eval_strategy=train_cfg["evaluation_strategy"],
        save_strategy=train_cfg["save_strategy"],
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=train_cfg.get("bf16", False),
        fp16=train_cfg.get("fp16", False),
        optim="adamw_8bit",
        report_to="wandb",
        run_name=run_name,
        dataloader_pin_memory=True,
    )

    # Create trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_data,
        eval_dataset=val_data,
        args=training_args,
        dataset_text_field="text",
        max_seq_length=model_cfg["max_seq_length"],
        packing=True,
    )

    # Train
    print("\\n🏋️ Starting training...")
    trainer.train()

    # Save
    print(f"\\nSaving to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Log completion
    wandb.log({"status": "complete"})
    wandb.finish()

    print("=" * 60)
    print(f"DONE: {model_key}")
    print(f"   Adapter saved to: {output_dir}")
    print("=" * 60)


if __name__ == "__main__":
    main()
'''

with open(f"{OUTPUT_DIR}/train.py", "w") as f:
    f.write(train_py)

print("Created train.py")

Created train.py


In [7]:
# CELL 7: Create requirements.txt

requirements = """# Core
torch>=2.1.0
transformers>=4.40.0
datasets>=2.18.0
accelerate>=0.28.0
peft>=0.10.0
trl>=0.8.0
bitsandbytes>=0.43.0

# Unsloth (faster training)
unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git

# Logging
wandb>=0.16.0

# Utils
sentencepiece
protobuf
scipy
einops

# GCS
google-cloud-storage
"""

with open(f"{OUTPUT_DIR}/requirements.txt", "w") as f:
    f.write(requirements)

print("Created requirements.txt")

Created requirements.txt


In [8]:
# CELL 8: Create setup.sh (Run once on Vast.ai)

setup_sh = """#!/bin/bash
# Run this ONCE when you first connect to Vast.ai instance

set -e
echo "========================================"
echo "Setting up Vast.ai instance"
echo "========================================"

# Install system packages
apt-get update
apt-get install -y git wget curl unzip

# Install Google Cloud SDK
if ! command -v gcloud &> /dev/null; then
    echo "Installing Google Cloud SDK..."
    curl -O https://dl.google.com/dl/cloudsdk/channels/rapid/downloads/google-cloud-cli-linux-x86_64.tar.gz
    tar -xf google-cloud-cli-linux-x86_64.tar.gz
    ./google-cloud-sdk/install.sh --quiet --path-update true
    source /root/google-cloud-sdk/path.bash.inc
    rm google-cloud-cli-linux-x86_64.tar.gz
fi

# Install Python packages
echo "Installing Python packages..."
pip install --upgrade pip
pip install -r requirements.txt

echo ""
echo "========================================"
echo "Setup complete!"
echo "========================================"
echo ""
echo "Next: Authenticate GCS with:"
echo "  gcloud auth activate-service-account --key-file=gcp-key.json"
echo ""
"""

with open(f"{OUTPUT_DIR}/setup.sh", "w") as f:
    f.write(setup_sh)

os.chmod(f"{OUTPUT_DIR}/setup.sh", 0o755)
print("Created setup.sh")

Created setup.sh


In [9]:
# Create run.sh (Run training + upload + shutdown)

run_sh = """#!/bin/bash
# Usage: ./run.sh <model_key>
# Example: ./run.sh falcon_h1_7b

set -e

MODEL_KEY=$1

if [ -z "$MODEL_KEY" ]; then
    echo "Usage: ./run.sh <model_key>"
    echo ""
    echo "Available models:"
    echo "  falcon_h1_7b        (theory)"
    echo "  qwen2_5_7b_4bit     (theory)"
    echo "  seed_coder_8b_4bit  (code)"
    echo "  acemath_7b          (math)"
    echo "  qwen2_5_math_7b_4bit (math)"
    exit 1
fi

echo "========================================"
echo "Starting training: $MODEL_KEY"
echo "========================================"

# Verify GCS auth
echo "Checking GCS authentication..."
gcloud auth activate-service-account --key-file=gcp-key.json
if ! gsutil ls gs://msda-llm-processed-data/ > /dev/null 2>&1; then
    echo "GCS auth failed!"
    exit 1
fi
echo "GCS OK"

# Verify W&B
if [ -z "$WANDB_API_KEY" ]; then
    echo "WANDB_API_KEY not set - logging may fail"
fi

# Run training
echo ""
echo "Training..."
python train.py --model_key $MODEL_KEY

# Check if training succeeded
if [ $? -eq 0 ]; then
    echo ""
    echo "Uploading adapter to GCS..."
    gsutil -m cp -r outputs/$MODEL_KEY/* gs://msda-llm-models/adapters/$MODEL_KEY/

    echo ""
    echo "========================================"
    echo "COMPLETE: $MODEL_KEY"
    echo "   Adapter: gs://msda-llm-models/adapters/$MODEL_KEY/"
    echo "========================================"
    echo ""
    echo "Shutting down in 30 seconds to save costs..."
    echo "   (Ctrl+C to cancel)"
    sleep 30
    sudo poweroff
else
    echo "Training failed!"
    exit 1
fi
"""

with open(f"{OUTPUT_DIR}/run.sh", "w") as f:
    f.write(run_sh)

os.chmod(f"{OUTPUT_DIR}/run.sh", 0o755)
print("Created run.sh")

Created run.sh


In [10]:
# Add GCP Key

# Upload your gcp-key.json file to Colab first!
# Then run this cell

from google.colab import files
import shutil

print("Please upload your GCP service account JSON key file...")
print("   (The file: msda-llm-finetuning-f82038d9000a.json)")
print("")

uploaded = files.upload()

# Get the uploaded filename
for filename in uploaded.keys():
    # Copy to output directory with standard name
    shutil.copy(filename, f"{OUTPUT_DIR}/gcp-key.json")
    print(f"Saved as: {OUTPUT_DIR}/gcp-key.json")

Please upload your GCP service account JSON key file...
   (The file: msda-llm-finetuning-f82038d9000a.json)



Saving msda-llm-finetuning-f82038d9000a.json to msda-llm-finetuning-f82038d9000a.json
Saved as: /content/vast_training/gcp-key.json


In [11]:
# Create ZIP and Download

import shutil

# Show what we're packaging
print("Contents of vast_training/:")
print("=" * 40)
for root, dirs, files_list in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for file in files_list:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        if size > 1024*1024:
            size_str = f"{size/1024/1024:.1f} MB"
        elif size > 1024:
            size_str = f"{size/1024:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"{subindent}{file} ({size_str})")

# Create zip
zip_path = "/content/vast_training"
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

zip_size = os.path.getsize(f"{zip_path}.zip") / (1024*1024)
print(f"\nCreated: vast_training.zip ({zip_size:.1f} MB)")

# Download
print("\nDownloading...")
files.download('/content/vast_training.zip')

Contents of vast_training/:
vast_training/
  run.sh (1.5 KB)
  gcp-key.json (2.3 KB)
  requirements.txt (315 B)
  setup.sh (1.1 KB)
  train.py (7.6 KB)
  configs/
    models.json (1.4 KB)
    training.json (2.1 KB)
  data/
    code_val.jsonl (2.1 MB)
    math_val.jsonl (1.9 MB)
    theory_val.jsonl (97.7 KB)
    theory_train.jsonl (869.2 KB)
    code_train.jsonl (18.4 MB)
    math_train.jsonl (17.3 MB)

Created: vast_training.zip (10.3 MB)

Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>